# GPU-native neural FSP validation

This notebook separates three effects on the exact 18-claim anchor:

1. the learned neural average;
2. the exact dense average of the same neural best-response sequence;
3. optionally, exact best response plus exact dense averaging.

The gap between 1 and 2 measures averaging error. The responder oracle gap measures neural BR error.

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.algo.neural_fsp import NeuralFSPTrainer
from liars_poker.algo.br_exact_dense_to_dense import best_response_dense
from liars_poker.eval.match_dense import evaluate_dense_response
from liars_poker.policies.action_conditioned import (
    compile_action_conditioned_to_dense,
    compile_action_conditioned_q_to_dense,
)
from liars_poker.policies.tabular_dense import DenseTabularPolicy, mix_dense

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name())

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips'),
    suit_symmetry=True,
)

OUTER_ITERATIONS = 5
RUN_EXACT_FSP_BASELINE = True

trainer = NeuralFSPTrainer(
    spec,
    average_state_hidden_sizes=(512, 512),
    average_action_hidden_sizes=(128, 128),
    average_embedding_dim=256,
    strategy_buffer_capacity=1_000_000,
    validation_buffer_capacity=20_000,
    validation_fraction=0.02,
    strategy_batch_size=4096,
    strategy_train_steps=100,
    strategy_learning_rate=1e-3,
    br_kwargs={
        'state_hidden_sizes': (512, 512),
        'action_hidden_sizes': (128, 128),
        'embedding_dim': 256,
        'replay_capacity': 1_000_000,
        'batch_size': 4096,
        'learning_rate': 1e-3,
        'train_steps': 100,
        'warmup_transitions': 20_000,
        'epsilon_start': 0.25,
        'epsilon_end': 0.05,
        'epsilon_decay_decisions': 250_000,
        'rollouts_per_action': 1,
        'fused_optimizer': True,
    },
    device=device,
    seed=7,
)

RUN_KWARGS = dict(
    br_iterations=100,
    br_episodes_per_role=512,
    br_rollout_batch_size=256,
    strategy_episodes_per_role=512,
    strategy_collection_batch_size=128,
    strategy_train_steps=100,
)

initial_dense = compile_action_conditioned_to_dense(
    trainer.average_policy(), batch_size=65_536,
)
neural_generated_average = initial_dense
exact_fsp_average = DenseTabularPolicy(spec)
print('claims:', len(initial_dense.rules.claims))

In [ ]:
def exploitability(policy):
    _, meta = best_response_dense(spec, policy, store_state_values=False)
    p_first, p_second = meta['computer'].exploitability()
    return p_first + p_second - 1.0

def mean_strategy_tv(first, second):
    legal = first.legal_mask[:, None, :]
    tv = 0.5 * np.abs(first.S - second.S) * legal
    return float(tv.sum(axis=2).mean())

rows = []
wall_start = time.perf_counter()
for outer in range(1, OUTER_ITERATIONS + 1):
    frozen_average = compile_action_conditioned_to_dense(
        trainer.average_policy(), batch_size=65_536,
    )
    _, frozen_meta = best_response_dense(
        spec, frozen_average, store_state_values=False,
    )
    exact_first, exact_second = frozen_meta['computer'].exploitability()
    exact_br_ceiling = exact_first + exact_second - 1.0

    record = trainer.run_iteration(**RUN_KWARGS)
    br_dense = compile_action_conditioned_q_to_dense(
        trainer.best_response_policy(), batch_size=65_536,
    )
    found_first, found_second = evaluate_dense_response(
        spec, frozen_average, br_dense,
    )
    found_br = found_first + found_second - 1.0

    eta = 1.0 / (outer + 1.0)
    neural_generated_average = mix_dense(
        br_dense, neural_generated_average, eta,
    )
    learned_average = compile_action_conditioned_to_dense(
        trainer.average_policy(), batch_size=65_536,
    )

    learned_exploitability = exploitability(learned_average)
    generated_exploitability = exploitability(neural_generated_average)

    exact_baseline_exploitability = np.nan
    if RUN_EXACT_FSP_BASELINE:
        exact_br, _ = best_response_dense(
            spec, exact_fsp_average, store_state_values=False,
        )
        exact_fsp_average = mix_dense(exact_br, exact_fsp_average, eta)
        exact_baseline_exploitability = exploitability(exact_fsp_average)

    rows.append({
        'outer iteration': outer,
        'elapsed min': (time.perf_counter() - wall_start) / 60.0,
        'learned average exploitability': learned_exploitability,
        'exact generated average exploitability': generated_exploitability,
        'learned minus generated': learned_exploitability - generated_exploitability,
        'average strategy TV': mean_strategy_tv(learned_average, neural_generated_average),
        'exact FSP exploitability': exact_baseline_exploitability,
        'neural BR discovered exploitability': found_br,
        'exact BR ceiling': exact_br_ceiling,
        'neural BR oracle gap': exact_br_ceiling - found_br,
        'BR training s': record['br_s'],
        'average collection s': record['strategy_collect_s'],
        'average fitting s': record['strategy_fit_s'],
        'strategy records seen': sum(record['strategy_records_seen']),
    })
    print(
        f"outer={outer:2d} learned={learned_exploitability:.5f} "
        f"generated={generated_exploitability:.5f} "
        f"BR gap={exact_br_ceiling - found_br:.5f}"
    )

results = pd.DataFrame(rows)
results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
x = results['outer iteration']

axes[0].plot(x, results['learned average exploitability'], marker='o', label='learned neural average')
axes[0].plot(x, results['exact generated average exploitability'], marker='o', label='exact average of neural BRs')
if RUN_EXACT_FSP_BASELINE:
    axes[0].plot(x, results['exact FSP exploitability'], marker='o', label='exact BR + exact average')
axes[0].set_yscale('log')
axes[0].set_title('Average-policy exploitability')

axes[1].plot(x, results['neural BR oracle gap'], marker='o', label='neural BR oracle gap')
axes[1].plot(x, results['average strategy TV'], marker='o', label='learned/generated strategy TV')
axes[1].set_yscale('log')
axes[1].set_title('Where approximation error enters')

axes[2].plot(x, results['BR training s'], marker='o', label='BR training')
axes[2].plot(x, results['average collection s'], marker='o', label='average collection')
axes[2].plot(x, results['average fitting s'], marker='o', label='average fitting')
axes[2].set_title('Per-outer-iteration cost')

for ax in axes:
    ax.set_xlabel('Outer FSP iteration')
    ax.grid(alpha=0.3, which='both')
    ax.legend()
plt.tight_layout()

results.tail(1).T

In [ ]:
# Phase-boundary checkpoint and exact reload smoke test.
checkpoint_path = REPO_ROOT / 'artifacts' / 'neural_fsp_smoke_checkpoint.pt'
trainer.save_checkpoint(checkpoint_path)
reloaded = NeuralFSPTrainer.load_checkpoint(checkpoint_path, device=device)
print('saved iteration:', trainer.iteration)
print('loaded iteration:', reloaded.iteration)
print('records seen:', [buffer.seen for buffer in reloaded.strategy_buffers])